In [2]:
import cv2
from keras.utils import load_img
from keras.saving import load_model
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from segment_anything import sam_model_registry, SamPredictor
from skimage.measure import regionprops, regionprops_table
from tqdm import trange, tqdm

import segmenteverygrain as seg

from tqdm import trange
%matplotlib inline

In [3]:
# checking if python is using the correct chip architecture
!python -c "import platform; print(platform.machine())"

arm64


In [4]:
# checking if the GPU is available to tensorflow
import tensorflow as tf
devices = tf.config.list_physical_devices()
print("\nDevices: ", devices)


Devices:  [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [5]:
from keras.optimizers import Adam
model = seg.Unet() # create model
model.compile(optimizer=Adam(), loss=seg.weighted_crossentropy, metrics=["accuracy"])

In [ ]:
# Alumina-only training set (no rock/sandstone data):
#   - prac7 etched alumina (3 pairs) and Wilson polished alumina (6 pairs)
input_dir = "alumina_training_data/"
patch_dir = "alumina_patches/"
image_dir, mask_dir = seg.patchify_training_data(input_dir, patch_dir)

100%|██████████| 89/89 [00:02<00:00, 39.11it/s]


In [10]:
# SAM3 auto-annotated clean images (89 pairs from sam3_mass_annotate.py).
# Masks use 1=interior, 2=boundary, 3=ignore; tf.one_hot(3, depth=3) -> [0,0,0], so
# weighted_crossentropy gives ignore pixels zero weight and the loss skips them.
# create_train_val_test_data expects the 256x256 *patch* folders that
# patchify_training_data produces -- not the folder of full-size image/mask pairs.
input_dir = "testcleanimages_sam3_annotated/images_and_masks/"
patch_dir = "sam3_annotated_patches/"
image_dir, mask_dir = seg.patchify_training_data(input_dir, patch_dir)

100%|██████████| 89/89 [00:02<00:00, 36.53it/s]


In [11]:
train_dataset, val_dataset, test_dataset = seg.create_train_val_test_data(image_dir, mask_dir, augmentation=True)

In [13]:
model = seg.create_and_train_model(train_dataset, val_dataset, test_dataset, epochs=200)

Epoch 1/200
50/50 ━━━━━━━━━━━━━━━━━━━━ 137s 3s/step - accuracy: 0.4293 - loss: 1.3939 - val_accuracy: 0.5897 - val_loss: 1.5112
Epoch 2/200
50/50 ━━━━━━━━━━━━━━━━━━━━ 148s 3s/step - accuracy: 0.6310 - loss: 1.0247 - val_accuracy: 0.6158 - val_loss: 1.5976
Epoch 3/200
50/50 ━━━━━━━━━━━━━━━━━━━━ 155s 3s/step - accuracy: 0.6576 - loss: 0.9653 - val_accuracy: 0.6160 - val_loss: 1.4336
Epoch 4/200
50/50 ━━━━━━━━━━━━━━━━━━━━ 160s 3s/step - accuracy: 0.6674 - loss: 0.9450 - val_accuracy: 0.6147 - val_loss: 1.3765
Epoch 5/200
50/50 ━━━━━━━━━━━━━━━━━━━━ 167s 3s/step - accuracy: 0.6760 - loss: 0.9355 - val_accuracy: 0.6323 - val_loss: 1.3620
Epoch 6/200
50/50 ━━━━━━━━━━━━━━━━━━━━ 169s 3s/step - accuracy: 0.6827 - loss: 0.9285 - val_accuracy: 0.6318 - val_loss: 1.1112
Epoch 7/200
50/50 ━━━━━━━━━━━━━━━━━━━━ 172s 3s/step - accuracy: 0.6825 - loss: 0.9184 - val_accuracy: 0.6377 - val_loss: 1.0733
Epoch 8/200
50/50 ━━━━━━━━━━━━━━━━━━━━ 181s 4s/step - accuracy: 0.6865 - loss: 0.9160 - val_accuracy: 0.

KeyboardInterrupt: 

In [ ]:
# save model
model.save('./models/seg_model_alumina_sam_annotated.keras')

In [ ]:
# load model (same path the save cell above writes to)
from keras.saving import load_model
loaded_model = load_model("./models/seg_model_alumina_sam_annotated.keras",
                          custom_objects={'weighted_crossentropy': seg.weighted_crossentropy})